# 39. Does the budget finding transfer to the encoded CatBoost?

**One variable against ledger row 26** (`catboost_te`, CV 0.966915): the number of boosting
iterations. Same nested target and frequency encoder, fingerprinted rather than assumed. Same
36 features, same 3 native categoricals, same `learning_rate=0.05`, same seed, same folds,
same `thread_count=6`.

## Why this run exists

Rows 78 to 84 found our raw-frame CatBoost undertrained by **+0.002622** at a bracketed optimum
of 10,000 iterations, against the 2,000 it had been given. That 2,000 came from the LightGBM
budget convention, `learning_rate * n_estimators = 100`, and was never fitted to CatBoost.

**Row 26 was given exactly the same inherited budget for exactly the same reason.** So were
rows 31 to 32 and 66 to 67. Every CatBoost number in this ledger sits at 2,000 iterations, and
none of them was chosen for CatBoost. If the raw-frame result transfers, row 26 at 0.966915
moves to roughly 0.9695, which would be **the best single model in this repo** by a clear
margin over `xgb_te` at 0.967099.

That is a big enough claim that it gets a run rather than an extrapolation. It may well not
transfer: the encoded frame gives CatBoost 24 columns of pre-computed target statistics, which
is precisely the information its own boosting has to spend iterations discovering on the raw
frame. **A representation that hands the model more up front should need fewer rounds, not
more**, so there is a real mechanism pointing the other way.

## The CatBoost auto-retuning trap, and why both parameters are pinned

Established 2026-08-21 and written up. CatBoost silently selects
`leaf_estimation_iterations` and `max_ctr_complexity` **from the value of `iterations`**: 1 and
1 up to 150, 10 and 4 from 200 upward. Sweeping iterations on defaults therefore moves three
things at once.

Row 26 ran at 2,000 and so received 10 and 4. Both are **pinned** at those values here, which
makes `iterations` the only variable and leaves the k=2000 arm an exact reproduction of row 26.
Row 26 does pass `cat_features`, so `max_ctr_complexity` is live here and pinning it matters.

## The design

One fit per fold at `N_MAX`, every checkpoint read off it with `predict_proba(ntree_end=k)`.
Boosting is sequential, so trees 1 to k of a long model are the k-iteration model. The contract
is asserted at bench scale before the full run, on a pair that does not straddle the auto-tune
threshold. No `eval_set`, so no early stopping and no path for the validation fold to reach
the fit.

**Checkpoints, pre-registered: `2000, 3000, 4000, 6000, 8000`.**

`N_MAX` stops at 8,000 rather than the 12,000 rows 78 to 84 used, because row 26 cost 80
minutes for five folds at 2,000 and this frame is more expensive per iteration than the raw
one. On the raw curve 8,000 captured 98.5 percent of the gain available at the 10,000 peak, so
the grid is short but not blind. If the curve is still climbing at 8,000 that is a result too,
and it says to extend rather than to stop.

**Fold vectors are written incrementally.** Rows 78 to 84 sat in the Kaggle queue for about six
hours before running, so wall-clock risk here is real, and a run killed at fold 4 should leave
four usable folds behind rather than nothing.

## The prediction, written before the run

**I predict the transfer is real but smaller than +0.002622**, somewhere in +0.0005 to +0.0020,
landing row 26's configuration between 0.9674 and 0.9689. The mechanism argument above says the
encoded frame needs fewer extra rounds than the raw one, and the raw frame had further to
travel because it started from a worse place.

**I predict it beats `xgb_te` at 0.967099 and becomes the best single model here.** That needs
only +0.00019.

The honest case against, and it is the one I would bet against myself with: 24 of the 36 columns
are already smoothed target statistics, so CatBoost's own ordered statistics have much less left
to find, and the whole gain could be inside noise. If the curve is flat, the correct reading is
that the budget deficit was a property of the raw representation and not of CatBoost, and rows
26, 31, 32, 66 and 67 all stand as recorded.

## What this decides

Nothing about the stack. It writes one member vector per checkpoint. Membership is a separate
notebook and a separate ledger row, per the rule that has held since row 24. No submission csv.

In [ ]:
# One flag. The run always goes top to bottom on Kaggle.
SMOKE = True

SEED = 42
N_INNER = 5
SMOOTH = 10.0

# Row 26's configuration, held fixed. Only the iteration count is swept.
LR = 0.05
PROBE_FOLD = 0

# PINNED at the values CatBoost auto-selects at 2,000 iterations, which is what row 26
# received. Without this, sweeping `iterations` also moves both of these. See this repo,
# 2026-08-21. Row 26 passes cat_features, so max_ctr_complexity is live here.
LEAF_EST_ITERS = 10
MAX_CTR_COMPLEXITY = 4

# Fixed rather than -1, matching row 26. deterministic results are pinned for a
# given thread count, not across thread counts, and 06 measured CatBoost bit-identical at 6.
THREADS = 6

CHECKPOINTS = [2000, 3000, 4000, 6000, 8000]
N_MAX = max(CHECKPOINTS)

# Above the auto-tune threshold at both ends, so the contract check does not straddle it.
BENCH_EST = 400

# A go/no-go gate checked ONCE before the run starts. It is not a timeout and cannot stop a
# run already going; that gap is recorded. Incremental saving below is the
# mitigation that actually helps.
MAX_HOURS = 9.0

BASELINE_NAME, BASELINE_CV, BASELINE_ROW = "catboost_te", 0.966915, "row 26 catboost_te"
EXPECTED_FOLD_SHA = "ec282b0968059676"
EXPECTED_ENCODER_FP = "0642e41750ef8bab"
EXPECTED_LEAK2 = 8.1e-05
EXPECTED_PRIOR_SHIFT = 1.3e-04

print(f"SMOKE = {SMOKE}   lr {LR}   checkpoints {CHECKPOINTS}")
print(f"one fit per fold at {N_MAX:,} iterations, curve read via ntree_end")
print(f"pinned: leaf_estimation_iterations={LEAF_EST_ITERS}, "
      f"max_ctr_complexity={MAX_CTR_COMPLEXITY}, thread_count={THREADS}")

## Stage 1. Data, folds, leak checklist

In [ ]:
import ast
import gc
import hashlib
import time
from pathlib import Path

import catboost as cb
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import KFold, StratifiedKFold

KAG = Path("/kaggle/input")
ON_KAGGLE = KAG.exists()
LOCAL = next((b for b in [Path.cwd(), *Path.cwd().parents]
              if (b / "data" / "raw" / "train.csv").exists()), None)


def locate(name):
    if ON_KAGGLE:
        hits = sorted(KAG.rglob(name))
        if hits:
            return hits[0]
    if LOCAL is not None:
        for d in ("data/raw", "artifacts/oof", "notebooks", "submissions"):
            p = LOCAL / d / name
            if p.exists():
                return p
    raise FileNotFoundError(name)


OUT = Path("/kaggle/working") if ON_KAGGLE else LOCAL / "artifacts" / "oof"
print(f"running {'on Kaggle' if ON_KAGGLE else 'locally'}, writing to {OUT}")
print(f"catboost {cb.__version__}, pandas {pd.__version__}, numpy {np.__version__}")

train_full = pd.read_csv(locate("train.csv"))
test = pd.read_csv(locate("test.csv"))
TARGET = "addicted_label"
CAT = ["gender", "stress_level", "academic_work_impact"]
COLS = [c for c in train_full.columns if c not in ("id", TARGET)]

checks = {
    "id is not a feature": "id" not in COLS,
    "target is not a feature": TARGET not in COLS,
    "train and test ids do not overlap":
        not (set(train_full["id"]) & set(test["id"])),
    "train and test feature lists match":
        COLS == [c for c in test.columns if c != "id"],
}
for name, ok in checks.items():
    print(f"  [{'ok' if ok else 'FAIL'}] {name}")
LEAK_OK = all(checks.values())

if SMOKE:
    ROW_IDX = np.sort(train_full.sample(20000, random_state=0).index.to_numpy())
    train = train_full.loc[ROW_IDX].reset_index(drop=True)
    test = test.head(5000).reset_index(drop=True)
    CHECKPOINTS = [50, 100, 200]
    N_MAX, BENCH_EST = 200, 50
    THREADS = 1
else:
    ROW_IDX = np.arange(len(train_full))
    train = train_full

y = train[TARGET].to_numpy()
X = train[COLS].copy()
X_test = test[COLS].copy()

folds = np.full(len(train), -1, dtype=np.int64)
for i, (_, va) in enumerate(StratifiedKFold(5, shuffle=True,
                                            random_state=SEED).split(train, y)):
    folds[va] = i

sha = hashlib.sha256(folds.tobytes()).hexdigest()[:16]
ALIGNED = sha == EXPECTED_FOLD_SHA
print()
print(f"rows {len(train):,}   target rate {y.mean():.6f}")
print(f"fold sizes {np.bincount(folds).tolist()}")
print(f"fold sha {sha}  expected {EXPECTED_FOLD_SHA}")
if SMOKE:
    print("SMOKE: subsampled, so the sha is EXPECTED to differ. Not a check.")
else:
    print("fold alignment: VERIFIED" if ALIGNED else
          "fold alignment: MISMATCH - the OOF from this run is not blendable")

## Stage 2. The encoder, fingerprinted rather than trusted

Copied verbatim from `34_xgb_depth.ipynb`, which copied it from `13_target_encoding.ipynb`.
The four functions are checksummed through `ast.unparse` and compared against `13`, because a
silently different encoder would make this two variables rather than one and would not show up
anywhere in the score. Expected `0642e41750ef8bab`, the value rows 26 and 33 recorded.

In [ ]:
def _stats(levels, yy, prior):
    """Smoothed target mean and level frequency, fit only on the rows given."""
    df = pd.DataFrame({"v": levels, "y": yy})
    g = df.groupby("v", dropna=False, observed=True)["y"].agg(["sum", "count"])
    mean = (g["sum"] + prior * SMOOTH) / (g["count"] + SMOOTH)
    return mean, g["count"] / len(df)


def _apply(levels, mean, freq, prior):
    s = pd.Series(levels)
    m = s.map(mean).to_numpy(dtype=np.float64)
    f = s.map(freq).to_numpy(dtype=np.float64)
    # A level unseen while fitting falls back to the prior and zero mass.
    return np.nan_to_num(m, nan=prior), np.nan_to_num(f, nan=0.0)


def encode_fold(Xf, yy, tr, va, Xt=None, seed=SEED):
    """Encodings for ONE outer fold: (train, valid, test)."""
    prior = float(yy[tr].mean())
    e_tr, e_va, e_te = {}, {}, {}
    splits = list(KFold(N_INNER, shuffle=True, random_state=seed).split(tr))

    for c in COLS:
        lv = Xf[c].to_numpy()
        # Fit once on the whole training portion, for validation and test rows.
        mean, freq = _stats(lv[tr], yy[tr], prior)
        e_va[f"te_{c}"], e_va[f"fq_{c}"] = _apply(lv[va], mean, freq, prior)
        if Xt is not None:
            e_te[f"te_{c}"], e_te[f"fq_{c}"] = _apply(Xt[c].to_numpy(), mean,
                                                      freq, prior)
        # Training rows get inner out-of-fold values.
        tm, tf = np.empty(len(tr)), np.empty(len(tr))
        for itr, iva in splits:
            im, if_ = _stats(lv[tr[itr]], yy[tr[itr]], prior)
            tm[iva], tf[iva] = _apply(lv[tr[iva]], im, if_, prior)
        e_tr[f"te_{c}"], e_tr[f"fq_{c}"] = tm, tf

    return (pd.DataFrame(e_tr), pd.DataFrame(e_va),
            pd.DataFrame(e_te) if Xt is not None else None)


def build(Xf, yy, tr, va, Xt=None, seed=SEED):
    d_tr, d_va, d_te = encode_fold(Xf, yy, tr, va, Xt, seed)
    Xtr = pd.concat([Xf.iloc[tr].reset_index(drop=True), d_tr], axis=1)
    Xva = pd.concat([Xf.iloc[va].reset_index(drop=True), d_va], axis=1)
    Xte = None if Xt is None else pd.concat(
        [Xt.reset_index(drop=True), d_te], axis=1)
    return Xtr, Xva, Xte


ENCODER_FNS = ("_stats", "_apply", "encode_fold", "build")


def fingerprint(src):
    """Semantic checksum of the encoder functions inside a block of source."""
    body = ast.parse(src).body
    parts = [ast.unparse(n) for n in body
             if isinstance(n, ast.FunctionDef) and n.name in ENCODER_FNS]
    if len(parts) != len(ENCODER_FNS):
        return None
    return hashlib.sha256("\n".join(parts).encode()).hexdigest()[:16]


import inspect

mine = fingerprint("\n".join(inspect.getsource(f)
                             for f in (_stats, _apply, encode_fold, build)))

theirs = None
try:
    src13 = locate("13_target_encoding.ipynb")
except FileNotFoundError:
    src13 = None
if src13 is not None:
    import json as _json
    for c in _json.loads(src13.read_text(encoding="utf-8"))["cells"]:
        if c["cell_type"] == "code" and "def encode_fold" in "".join(c["source"]):
            theirs = fingerprint("".join(c["source"]))
            break

ENCODER_MATCH = mine is not None and mine == theirs
print(f"encoder fingerprint here        : {mine}")
print(f"encoder fingerprint in 13       : {theirs}")
print(f"rows 26 and 33 recorded         : 0642e41750ef8bab")
print("encoder: IDENTICAL to row 17's" if ENCODER_MATCH else
      "encoder: DIFFERS from 13 (or 13 not found) - this is NOT one variable")
print()
print(f"{len(COLS)} raw columns -> {len(COLS) * 3} features after encoding")

### The leak checks, by execution

The same checks `13` ran, on the same encoder, so their numbers are directly comparable. The
encoder is the only thing in this notebook capable of leaking, and it is fit inside the fold
loop and inside an inner `KFold` within that.

In [ ]:
_tr = np.where(folds != 0)[0]
_va = np.where(folds == 0)[0]
d_tr0, d_va0, _ = encode_fold(X, y, _tr, _va)

# 1. A validation row's own target must never reach its own encoding.
y1 = y.copy()
y1[_va] = 1 - y1[_va]
_, d_va1, _ = encode_fold(X, y1, _tr, _va)
leak1 = max(np.abs(d_va0[f"te_{c}"] - d_va1[f"te_{c}"]).max() for c in COLS)

# 2. A training row's own target must never reach its own inner encoding. A small
# residual is expected and is not a leak: `prior` is the training-portion mean.
_, iva0 = list(KFold(N_INNER, shuffle=True, random_state=SEED).split(_tr))[0]
pick = iva0[:200]
y2 = y.copy()
y2[_tr[pick]] = 1 - y2[_tr[pick]]
d_tr2, _, _ = encode_fold(X, y2, _tr, _va)
leak2 = max(np.abs(d_tr0[f"te_{c}"].to_numpy()[pick]
                   - d_tr2[f"te_{c}"].to_numpy()[pick]).max() for c in COLS)
prior_shift = abs(float(y2[_tr].mean()) - float(y[_tr].mean()))

# 3. The encoding MUST move when targets it is allowed to see change.
y3 = y.copy()
y3[_tr] = 1 - y3[_tr]
_, d_va3, _ = encode_fold(X, y3, _tr, _va)
live = max(np.abs(d_va0[f"te_{c}"] - d_va3[f"te_{c}"]).max() for c in COLS)

print(f"1. flip all validation targets -> change in their encoding: {leak1:.3e}")
print(f"2. flip 200 training rows -> change in their own encoding:  {leak2:.3e}")
print(f"   prior moved {prior_shift:.3e}, and these should track each other")
print(f"3. flip all training targets -> change in val encoding:     {live:.3e}")

CLEAN = leak1 == 0 and leak2 < 10 * max(prior_shift, 1e-9) and live > 0.1
print()
print("LEAK CHECKS: PASS" if CLEAN else "LEAK CHECKS: FAILED - do not log this run")
if not SMOKE:
    print(f"13 recorded leak2 {EXPECTED_LEAK2:.1e} against a prior shift of "
          f"{EXPECTED_PRIOR_SHIFT:.1e}; this run gives {leak2:.1e} and "
          f"{prior_shift:.1e}")

del d_tr0, d_va0, d_va1, d_tr2, d_va3, y1, y2, y3
gc.collect()

## Stage 3. Bench, determinism, the `ntree_end` contract, and the projection

In [ ]:
def to_cb(df):
    """CatBoost wants categoricals as strings with no NaN. Row 26's helper, unchanged."""
    d = df.copy()
    for c in CAT:
        d[c] = d[c].astype("object").fillna("__NA__").astype(str)
    return d


def make_cat(n_est):
    return cb.CatBoostClassifier(
        iterations=n_est, learning_rate=LR, random_seed=SEED,
        leaf_estimation_iterations=LEAF_EST_ITERS,
        max_ctr_complexity=MAX_CTR_COMPLEXITY,
        thread_count=THREADS, allow_writing_files=False, verbose=0,
    )


def encoded_fold(fold, want_test=False):
    tr = np.where(folds != fold)[0]
    va = np.where(folds == fold)[0]
    Xtr, Xva, Xte = build(X, y, tr, va, X_test if want_test else None)
    Xtr, Xva = to_cb(Xtr), to_cb(Xva)
    idx = [Xtr.columns.get_loc(c) for c in CAT]
    return tr, va, Xtr, Xva, (to_cb(Xte) if want_test else None), idx


def hhmm(s):
    s = int(s)
    return f"{s // 3600}h {s % 3600 // 60:02d}m" if s >= 3600 else f"{s // 60}m {s % 60:02d}s"


t_enc = time.time()
tr0, va0, Xtr0, Xva0, Xte0, CAT_IDX = encoded_fold(PROBE_FOLD, want_test=True)
enc_secs = time.time() - t_enc
print(f"encoding one fold took {hhmm(enc_secs)}, {Xtr0.shape[1]} features, "
      f"cat_features at {CAT_IDX}")

t0 = time.time()
m1 = make_cat(BENCH_EST)
m1.fit(Xtr0, y[tr0], cat_features=CAT_IDX)
t1 = time.time() - t0
p1 = m1.predict_proba(Xva0)[:, 1]

m2 = make_cat(BENCH_EST)
m2.fit(Xtr0, y[tr0], cat_features=CAT_IDX)
drift = float(np.max(np.abs(p1 - m2.predict_proba(Xva0)[:, 1])))
print(f"bench {BENCH_EST} iters: AUC {roc_auc_score(y[va0], p1):.6f} in {hhmm(t1)}, "
      f"repeat drift {drift:.3e} {'OK' if drift == 0.0 else 'NOT DETERMINISTIC'}")

half = max(2, BENCH_EST // 2)
m_half = make_cat(half)
m_half.fit(Xtr0, y[tr0], cat_features=CAT_IDX)
slice_drift = float(np.max(np.abs(m_half.predict_proba(Xva0)[:, 1]
                                  - m1.predict_proba(Xva0, ntree_end=half)[:, 1])))
print(f"ntree_end contract at k={half}: max |direct - sliced| = {slice_drift:.3e} "
      f"{'OK' if slice_drift == 0.0 else 'BROKEN, the curve below would be meaningless'}")
NTREE_OK = slice_drift == 0.0

tp = time.time()
_ = m1.predict_proba(Xva0, ntree_end=BENCH_EST)[:, 1]
_ = m1.predict_proba(Xte0, ntree_end=BENCH_EST)[:, 1]
per_pred_iter = (time.time() - tp) / BENCH_EST

del m1, m2, m_half, Xtr0, Xva0, Xte0
gc.collect()

per_iter = t1 / BENCH_EST
train_secs = per_iter * N_MAX * 5
pred_secs = per_pred_iter * sum(CHECKPOINTS) * 5
enc_total = enc_secs * 5
projected = train_secs + pred_secs + enc_total
print(f"\nprojected encoding:   {hhmm(enc_total)} for 5 folds")
print(f"projected training:   {hhmm(train_secs)} for 5 folds x {N_MAX:,} iterations")
print(f"projected prediction: {hhmm(pred_secs)} for {len(CHECKPOINTS)} checkpoints")
print(f"projected TOTAL:      {hhmm(projected)}")
GO = projected < MAX_HOURS * 3600 or SMOKE
print("within budget" if GO else
      f"OVER the {MAX_HOURS}h guard, not starting. Drop checkpoints or N_MAX and re-push.")

## Stage 4. The run

Fold vectors are saved after every fold. A run killed by the wall clock then leaves usable
folds behind instead of nothing, which matters because the previous kernel spent about six
hours queued before it started.

In [ ]:
assert CLEAN and LEAK_OK, "leak checks failed"
assert ENCODER_MATCH, "encoder does not match 13, this would not be one variable"
assert NTREE_OK, "ntree_end contract failed"
assert GO, "over the time guard"
if not SMOKE:
    assert ALIGNED, "fold sha mismatch"

pre = "SMOKE_" if SMOKE else ""
oof = {k: np.zeros(len(train)) for k in CHECKPOINTS}
tst = {k: np.zeros((5, len(test))) for k in CHECKPOINTS}
per_fold = {k: [] for k in CHECKPOINTS}
done_folds = []

t_start = time.time()
for f in range(5):
    tr, va, Xtr, Xva, Xte, idx = encoded_fold(f, want_test=True)
    tf = time.time()
    m = make_cat(N_MAX)
    m.fit(Xtr, y[tr], cat_features=idx)
    line = []
    for k in CHECKPOINTS:
        pv = m.predict_proba(Xva, ntree_end=k)[:, 1]
        oof[k][va] = pv
        tst[k][f] = m.predict_proba(Xte, ntree_end=k)[:, 1]
        a = float(roc_auc_score(y[va], pv))
        per_fold[k].append(a)
        line.append(f"{k}:{a:.6f}")
    del m, Xtr, Xva, Xte
    gc.collect()
    done_folds.append(f)

    # Incremental save. Partial by construction until the last fold lands.
    for k in CHECKPOINTS:
        np.save(OUT / f"{pre}PARTIAL_cat_te_n{k}_oof.npy", oof[k])
        np.save(OUT / f"{pre}PARTIAL_cat_te_n{k}_test.npy", tst[k])
    np.save(OUT / f"{pre}PARTIAL_folds_done.npy", np.array(done_folds))

    done = time.time() - t_start
    print(f"fold {f} in {hhmm(time.time() - tf)}  " + "  ".join(line))
    print(f"         elapsed {hhmm(done)}, about {hhmm(done / (f + 1) * (4 - f))} left")

print(f"\nall folds done in {hhmm(time.time() - t_start)}")

In [ ]:
cv = {k: float(np.mean(per_fold[k])) for k in CHECKPOINTS}
sd = {k: float(np.std(per_fold[k])) for k in CHECKPOINTS}

print("The curve. k=2000 is row 26 refit, not a treatment.\n")
print(f"{'iterations':>11} {'CV':>10} {'sd':>9} {'vs k=2000':>11} {'folds won':>10}")
base = np.array(per_fold[CHECKPOINTS[0]])
for k in CHECKPOINTS:
    d = np.array(per_fold[k]) - base
    w = "" if k == CHECKPOINTS[0] else f"{int((d > 0).sum())}/5"
    v = "" if k == CHECKPOINTS[0] else f"{d.mean():+11.6f}"
    print(f"{k:>11,} {cv[k]:10.6f} {sd[k]:9.6f} {v:>11} {w:>10}")

repro = cv[CHECKPOINTS[0]] - BASELINE_CV
print(f"\nreproduction of {BASELINE_ROW}: {cv[CHECKPOINTS[0]]:.6f} vs {BASELINE_CV:.6f}"
      f"  delta {repro:+.2e}")
if SMOKE:
    print("SMOKE: this did NOT run at full size, so the reproduction check DID NOT RUN.")
else:
    print("REPRODUCED" if abs(repro) < 1e-4 else "FAILED - every number above is void")

best_k = max(CHECKPOINTS, key=lambda k: cv[k])
d = np.array(per_fold[best_k]) - base
sdp = d.std(ddof=1)
print(f"\nbest checkpoint {best_k:,} at {cv[best_k]:.6f}")
print(f"  paired vs k=2000: {d.mean():+.6f}, sd {sdp:.6f}, "
      f"{int((d > 0).sum())}/5 folds"
      + (f", t(4)={d.mean() / (sdp / np.sqrt(5)):.2f}" if sdp > 0 else ""))
print(f"  optimum interior to the grid? "
      f"{'yes, bracketed' if best_k not in (CHECKPOINTS[0], CHECKPOINTS[-1]) else 'NO, at an edge - the grid should be extended'}")
print(f"\nthe two claims this run was built to test:")
print(f"  transfers at all (raw frame got +0.002622): {d.mean():+.6f}")
print(f"  beats xgb_te 0.967099, best single here:    "
      f"{cv[best_k]:.6f} {'YES' if cv[best_k] > 0.967099 else 'no'}")

In [ ]:
for k in CHECKPOINTS:
    np.save(OUT / f"{pre}cat_te_n{k}_oof.npy", oof[k])
    np.save(OUT / f"{pre}cat_te_n{k}_test.npy", tst[k].mean(axis=0))
    print(f"wrote {pre}cat_te_n{k}_oof.npy, {pre}cat_te_n{k}_test.npy")

print("\nledger lines:")
for k in CHECKPOINTS:
    print(f"  name    cat_te_n{k}\n  cv_mean {cv[k]:.6f}\n  cv_std  {sd[k]:.6f}")
print(f"\n  leak checks {'PASS' if CLEAN else 'FAILED'}, "
      f"encoder {EXPECTED_ENCODER_FP if ENCODER_MATCH else 'MISMATCH'}, "
      f"fold alignment {'verified' if ALIGNED else 'MISMATCH'}")
print("\nNo submission csv. Membership is decided in a separate notebook, which changes")
print("a different variable and gets its own ledger row.")